In [80]:
import pandas as pd
import glob
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

try:
    from xgboost import XGBRegressor
except ImportError as exc:
    raise ImportError(
        "XGBoost is required for this notebook. Install it with: pip install xgboost"
    ) from exc

pd.set_option('display.max_columns', None)

In [81]:
# folder path
path = r"D:\egg_price_procject\output\main_data\*.csv"
# get all csv files
files = glob.glob(path)

In [82]:
# getting all cities data in dic all_data
all_data = {}
for file in files:
    name = file.split("\\")[-1].split(".")[0]
    file_name = name
    name = pd.read_csv(file)
    all_data[file_name] = name
print(all_data.keys())
df = all_data['Barwala_main_data']

dict_keys(['Ajmer_main_data', 'Barwala_main_data', 'Delhi_main_data', 'Hyderabad_main_data', 'Indore_main_data', 'Varanasi_main_data'])


In [83]:
df.head()

,Category,City,Price,Day,Month,Year,Date,daily_change_pct,market_rating,lag_1,lag_7,rolling_mean_7,rolling_mean_14,rolling_mean_30,rolling_std_7,dayofweek,weekofyear,quarter,is_weekend,lag_3,lag_14,lag_30,date,tmax,prcp,festival_name,is_festival
0,NECC SUGGESTED EGG PRICES,Barwala,375.0,12,5,2016,2016-05-12,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,3,19,2,0,NaN,NaN,NaN,2016-05-12,37.3,0.0,NaN,NaN
1,NECC SUGGESTED EGG PRICES,Barwala,375.0,13,5,2016,2016-05-13,0.0,0,375.0,NaN,NaN,NaN,NaN,NaN,4,19,2,0,NaN,NaN,NaN,2016-05-13,39.5,0.5,NaN,NaN
2,NECC SUGGESTED EGG PRICES,Barwala,378.0,14,5,2016,2016-05-14,0.8,0,375.0,NaN,NaN,NaN,NaN,NaN,5,19,2,1,NaN,NaN,NaN,2016-05-14,40.9,0.0,NaN,NaN
3,NECC SUGGESTED EGG PRICES,Barwala,378.0,15,5,2016,2016-05-15,0.0,0,378.0,NaN,NaN,NaN,NaN,NaN,6,19,2,1,375.0,NaN,NaN,2016-05-15,41.1,0.0,NaN,NaN
4,NECC SUGGESTED EGG PRICES,Barwala,378.0,16,5,2016,2016-05-16,0.0,0,378.0,NaN,NaN,NaN,NaN,NaN,0,20,2,0,375.0,NaN,NaN,2016-05-16,41.3,0.2,NaN,NaN


In [84]:
df.shape

(3616, 27)

creating seperate model for each day

In [85]:
df  = all_data['Barwala_main_data']

In [87]:
df['Date'] = pd.to_datetime(df['Date'])

In [88]:
extract = ['City','Price','Date','market_rating','lag_30','rolling_mean_30','tmax','prcp','is_weekend','is_festival']

In [89]:
df_new= df[extract]
df_new['is_festival'] = df_new['is_festival'].notna().astype(int)

C:\Users\user\AppData\Local\Temp\ipykernel_27588\3582164350.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_new['is_festival'] = df_new['is_festival'].notna().astype(int)


In [90]:
df_new.head()

,City,Price,Date,market_rating,lag_30,rolling_mean_30,tmax,prcp,is_weekend,is_festival
0,Barwala,375.0,2016-05-12,0,NaN,NaN,37.3,0.0,0,0
1,Barwala,375.0,2016-05-13,0,NaN,NaN,39.5,0.5,0,0
2,Barwala,378.0,2016-05-14,0,NaN,NaN,40.9,0.0,1,0
3,Barwala,378.0,2016-05-15,0,NaN,NaN,41.1,0.0,1,0
4,Barwala,378.0,2016-05-16,0,NaN,NaN,41.3,0.2,0,0


In [91]:
df_new.dtypes

City                       object
Price                     float64
Date               datetime64[ns]
market_rating               int64
lag_30                    float64
rolling_mean_30           float64
tmax                      float64
prcp                      float64
is_weekend                  int64
is_festival                 int32
dtype: object

In [92]:
df_new = df_new.sort_values("Date")
df_new = df_new.reset_index(drop=True)

In [93]:
datasets = {}

for horizon in range(1, 8):

    temp_df = df_new.copy()
    temp_df['Date'] = pd.to_datetime(temp_df['Date'])

    # Create future target
    temp_df["target"] = (
        temp_df["Price"].shift(-horizon)
    )

    # Remove NaN rows
    temp_df = temp_df.dropna()

    # Store dataset
    datasets[horizon] = temp_df

    print(
        f"{horizon}-day dataset shape:",
        temp_df.shape
    )

1-day dataset shape: (3585, 11)
2-day dataset shape: (3584, 11)
3-day dataset shape: (3583, 11)
4-day dataset shape: (3582, 11)
5-day dataset shape: (3581, 11)
6-day dataset shape: (3580, 11)
7-day dataset shape: (3579, 11)


In [94]:
datasets.keys()

dict_keys([1, 2, 3, 4, 5, 6, 7])

In [95]:
datasets[7].head(3)

,City,Price,Date,market_rating,lag_30,rolling_mean_30,tmax,prcp,is_weekend,is_festival,target
30,Barwala,360.0,2016-06-11,0,375.0,355.6,37.8,0.5,1,0,380.0
31,Barwala,360.0,2016-06-12,0,375.0,355.1,38.8,0.0,1,0,383.0
32,Barwala,367.0,2016-06-13,1,378.0,354.6,40.1,0.3,0,0,386.0


In [96]:
datasets[1].head(3)

,City,Price,Date,market_rating,lag_30,rolling_mean_30,tmax,prcp,is_weekend,is_festival,target
30,Barwala,360.0,2016-06-11,0,375.0,355.6,37.8,0.5,1,0,360.0
31,Barwala,360.0,2016-06-12,0,375.0,355.1,38.8,0.0,1,0,367.0
32,Barwala,367.0,2016-06-13,1,378.0,354.6,40.1,0.3,0,0,373.0


In [97]:
datasets.keys()

dict_keys([1, 2, 3, 4, 5, 6, 7])

In [98]:
# give the vlaue of day gape model you want
df = datasets[4].copy()

In [99]:
df["Date"] = pd.to_datetime(df["Date"])

In [100]:
df = df.sort_values("Date")

In [101]:
df["lag_1"] = df["Price"].shift(1)
df["lag_3"] = df["Price"].shift(3)
df["lag_7"] = df["Price"].shift(7)
df["lag_14"] = df["Price"].shift(14)

In [102]:
df["rolling_mean_7"] = (
    df["Price"]
    .rolling(7)
    .mean()
)

df["rolling_mean_14"] = (
    df["Price"]
    .rolling(14)
    .mean()
)

df["rolling_std_7"] = (
    df["Price"]
    .rolling(7)
    .std()
)

In [103]:
df["dayofweek"] = df["Date"].dt.dayofweek
df["month"] = df["Date"].dt.month
df["quarter"] = df["Date"].dt.quarter
df["weekofyear"] = (
    df["Date"]
    .dt
    .isocalendar()
    .week
    .astype(int)
)

In [104]:
df["dayofyear"] = df["Date"].dt.dayofyear

df["sin_year"] = np.sin(
    2 * np.pi * df["dayofyear"] / 365
)

df["cos_year"] = np.cos(
    2 * np.pi * df["dayofyear"] / 365
)

In [105]:
df = df.dropna().reset_index(drop=True)

In [106]:
df.shape

(3568, 25)

In [107]:
split_date = "2025-01-01"

In [108]:
df.columns

Index(['City', 'Price', 'Date', 'market_rating', 'lag_30', 'rolling_mean_30',
       'tmax', 'prcp', 'is_weekend', 'is_festival', 'target', 'lag_1', 'lag_3',
       'lag_7', 'lag_14', 'rolling_mean_7', 'rolling_mean_14', 'rolling_std_7',
       'dayofweek', 'month', 'quarter', 'weekofyear', 'dayofyear', 'sin_year',
       'cos_year'],
      dtype='object')

In [ ]:
df.head()

In [109]:
train_df = df[df["Date"] < split_date]
test_df = df[df["Date"] >= split_date]

In [110]:
# target = "Price"
features= ["market_rating","lag_1","lag_3","lag_7","lag_14","lag_30","rolling_mean_7"
            ,"rolling_mean_14","rolling_mean_30","rolling_std_7","tmax","prcp","dayofweek","weekofyear",
            "quarter","is_weekend","is_festival"]
# len(features_train)

In [111]:
# features = [

#     "market_rating",

#     "lag_1",
#     "lag_3",
#     "lag_7",
#     "lag_14",

#     "rolling_mean_7",
#     "rolling_mean_14",
#     "rolling_std_7",

#     "dayofweek",
#     "month",
#     "quarter",
#     "weekofyear",

#     "dayofyear",
#     "sin_year",
#     "cos_year"
# ]

In [112]:
X_train = train_df[features]
y_train = train_df["target"]

X_test = test_df[features]
y_test = test_df["target"]

In [113]:
X_train.columns

Index(['market_rating', 'lag_1', 'lag_3', 'lag_7', 'lag_14', 'lag_30',
       'rolling_mean_7', 'rolling_mean_14', 'rolling_mean_30', 'rolling_std_7',
       'tmax', 'prcp', 'dayofweek', 'weekofyear', 'quarter', 'is_weekend',
       'is_festival'],
      dtype='object')

In [114]:
model = XGBRegressor(

    n_estimators=500,
    learning_rate=0.03,
    max_depth=5,

    subsample=0.8,
    colsample_bytree=0.8,

    random_state=42
)

In [115]:
# model.fit(X_train, y_train)

In [116]:
# Tune hyperparameters with time-series aware cross-validation.
# RandomizedSearchCV is faster than a full grid while still testing strong parameter combinations.
tscv = TimeSeriesSplit(n_splits=5)

xgb_base = XGBRegressor(
    objective="reg:squarederror",
    eval_metric="rmse",
    random_state=42,
    n_jobs=-1,
    tree_method="hist"
)

param_distributions = {
    "n_estimators": [300, 500, 800, 1000],
    "max_depth": [2, 3, 4, 5, 6],
    "learning_rate": [0.01, 0.03, 0.05, 0.08, 0.1],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "min_child_weight": [1, 3, 5, 7],
    "reg_alpha": [0, 0.01, 0.05, 0.1],
    "reg_lambda": [0.5, 1, 2, 5],
    "gamma": [0, 0.01, 0.05, 0.1]
}

search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_distributions,
    n_iter=60,
    scoring="neg_root_mean_squared_error",
    cv=tscv,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

search.fit(X_train, y_train)

best_model = search.best_estimator_
print("Best CV RMSE:", round(-search.best_score_, 3))
print("Best parameters:")
search.best_params_


Fitting 5 folds for each of 60 candidates, totalling 300 fits
Best CV RMSE: 27.55
Best parameters:


{'subsample': 0.8,
 'reg_lambda': 2,
 'reg_alpha': 0.05,
 'n_estimators': 800,
 'min_child_weight': 5,
 'max_depth': 2,
 'learning_rate': 0.01,
 'gamma': 0.1,
 'colsample_bytree': 1.0}

In [117]:
# # Predict on the untouched 2025 test period.
# test_results = test_data[["Date", target]].copy()
# test_results["Predicted_Price"] = best_model.predict(x_test)
# test_results["Error"] = test_results[target] - test_results["Predicted_Price"]
# test_results["Absolute_Error"] = test_results["Error"].abs()

# mae = mean_absolute_error(y_test, test_results["Predicted_Price"])
# rmse = root_mean_squared_error(y_test, test_results["Predicted_Price"])
# r2 = r2_score(y_test, test_results["Predicted_Price"])
# mape = np.mean(np.abs(test_results["Error"] / y_test)) * 100

# metrics = pd.DataFrame({
#     "Metric": ["MAE", "RMSE", "R2", "MAPE (%)"],
#     "Value": [mae, rmse, r2, mape]
# })
# metrics["Value"] = metrics["Value"].round(3)

# print(f"Mean absolute error: {mae:.2f} price points")
# print(f"Root mean squared error: {rmse:.2f} price points")
# print(f"R2 score: {r2:.3f}")
# print(f"Mean absolute percentage error: {mape:.2f}%")
# metrics


In [118]:
preds = best_model.predict(X_test)

In [119]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [120]:
mae = mean_absolute_error(y_test, preds)

rmse = np.sqrt(
    mean_squared_error(y_test, preds)
)

r2 = r2_score(y_test, preds)

In [121]:
print("MAE :", mae)
print("RMSE:", rmse)
print("R2  :", r2)

MAE : 21.026775739255832
RMSE: 27.87020497266013
R2  : 0.8791783974674107


In [122]:
result_df = pd.DataFrame({

    "Date": test_df["Date"],

    "Actual": y_test.values,

    "Predicted": preds
})

In [123]:
result_df.head(10)

,Date,Actual,Predicted
3075,2025-01-01,505.0,568.641846
3076,2025-01-02,505.0,553.673584
3077,2025-01-03,505.0,547.561035
3078,2025-01-04,515.0,547.640625
3079,2025-01-05,520.0,502.255096
3080,2025-01-06,520.0,504.962952
3081,2025-01-07,500.0,496.352661
3082,2025-01-08,495.0,505.525146
3083,2025-01-09,485.0,501.902618
3084,2025-01-10,485.0,508.158020


In [124]:
import plotly.express as px

In [125]:
fig = px.line(

    result_df,

    x="Date",

    y=["Actual", "Predicted"],

    title="Actual vs Predicted Prices"
)

fig.show()

In [126]:
importance_df = pd.DataFrame({

    "Feature": features,

    "Importance": best_model.feature_importances_
})

In [127]:
importance_df = (
    importance_df
    .sort_values(
        "Importance",
        ascending=False
    )
)

In [128]:
fig = px.bar(

    importance_df,

    x="Importance",

    y="Feature",

    orientation="h",

    title="Feature Importance"
)

fig.show()